# DLP Scoring Pipeline
Loads the trained CatBoost model and scores new events from the Delta table.
Writes predictions to `workspace.default.dlp_predictions`.
Runs daily after Data_Ingestion.ipynb via Databricks Workflows.

In [0]:
%pip install catboost numpy==1.26.4 --quiet

In [0]:
import os
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from pyspark.sql import functions as F
from datetime import datetime, timedelta

## 1. Configuration

In [0]:
SOURCE_TABLE      = "workspace.default.dlp_features_clean"
PREDICTIONS_TABLE = "workspace.default.dlp_predictions"

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = os.path.dirname(os.path.dirname(notebook_path))
MODEL_PATH = f"/Workspace{repo_root}/dlp_catboost_v1.cbm"

# Score events from the last 24 hours only
SCORING_WINDOW_HOURS = 24

print(f"Run date     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Source table : {SOURCE_TABLE}")
print(f"Output table : {PREDICTIONS_TABLE}")
print(f"Model path   : {MODEL_PATH}")

## 2. Define Feature Columns

In [0]:
LABEL_COL = "Risk_Label"

CATEGORICAL_COLS = [
    "ActionType",
    "file_extension",
    "Position",
]

NUMERICAL_COLS = [
    "is_high_risk_extension",
    "file_name_has_sensitive_keyword",
    "double_extension_check",
    "is_risky_target_domain",
    "is_first_time_domain",
    "is_after_hours",
    "day_of_week",
    "is_monday_or_friday",
    "is_high_risk_position",
    "user_upload_count_24h",
    "user_unique_domains_7d",
]

FEATURE_COLS = CATEGORICAL_COLS + NUMERICAL_COLS

## 3. Load New Events
Only scores events from the last 24 hours that haven't been scored yet.

In [0]:
cutoff = datetime.utcnow() - timedelta(hours=SCORING_WINDOW_HOURS)
cutoff_str = cutoff.strftime("%Y-%m-%dT%H:%M:%SZ")

features_df   = spark.table(SOURCE_TABLE)
new_events_df = features_df.filter(F.col("Timestamp") >= cutoff_str)

# Check if predictions table exists before trying to read it
table_exists = spark.sql(
    f"SHOW TABLES IN workspace.default LIKE 'dlp_predictions'"
).count() > 0

if table_exists:
    scored_ids    = spark.table(PREDICTIONS_TABLE).select("Action_ID")
    new_events_df = new_events_df.join(scored_ids, on="Action_ID", how="left_anti")
else:
    print("First run ")

n_new = new_events_df.count()
print(f"New events to score: {n_new}")

if n_new == 0:
    print("No new events to score. Exiting.")
    dbutils.notebook.exit("No new events")

## 4. Load Model

In [0]:
model = CatBoostClassifier()
model.load_model(MODEL_PATH)

## 5. Score New Events

In [0]:
# Convert to pandas for scoring
df = new_events_df.toPandas()

X = df[FEATURE_COLS].copy()
for c in CATEGORICAL_COLS:
    X[c] = X[c].astype(str)

# Generate predictions
df["predicted"]      = model.predict(X)
df["predicted_prob"] = model.predict_proba(X)[:, 1].round(4)
df["scored_at"]      = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
df["model_version"]  = "catboost_v1"

print(f"Scored {len(df)} events")
print(f"Predicted risk distribution:\n{df['predicted'].value_counts()}")
print()
print("High risk events flagged:")
flagged = df[df["predicted"] == 1][[
    "Action_ID", "AccountDisplayName", "ObjectName",
    "Target_Domain", "Position", "predicted_prob"
]].sort_values("predicted_prob", ascending=False)
print(flagged.to_string(index=False))

## 6. Write Predictions to Delta Table

In [0]:
# Select columns to store
output_cols = [
    "Action_ID", "Timestamp", "AccountDisplayName", "ObjectName",
    "Target_Domain", "Position", "Risk_Label",
    "predicted", "predicted_prob", "scored_at", "model_version"
]

predictions_spark_df = spark.createDataFrame(df[output_cols])
predictions_spark_df.write.mode("append").saveAsTable(PREDICTIONS_TABLE)

total_predictions = spark.table(PREDICTIONS_TABLE).count()
print(f"Predictions written successfully.")
print(f"Total rows in predictions table: {total_predictions}")

## 7. Summary

In [0]:
print(f"Run date          : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Events scored     : {len(df)}")
print(f"Flagged as risky  : {int(df['predicted'].sum())}")
print(f"Safe              : {int((df['predicted'] == 0).sum())}")
print(f"Avg risk prob     : {df['predicted_prob'].mean():.4f}")
print(f"Max risk prob     : {df['predicted_prob'].max():.4f}")
print(f"Model version     : catboost_v1")

In [0]:
display(spark.table(PREDICTIONS_TABLE).orderBy(F.col("predicted_prob").desc()))

In [0]:
display(
    spark.table(PREDICTIONS_TABLE)
    .where(
    (F.col("predicted") == 0) &
    (F.col("Risk_Label") == 1)
    )
)
